In [2]:
##Membuat spark session
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/11 22:17:32 WARN Utils: Your hostname, LOQ-CEED resolves to a loopback address: 127.0.1.1; using 192.168.0.120 instead (on interface wlan0)
26/09/11 22:17:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 22:17:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [32]:
##Membaca dataset dan eksplorasi awal
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/TransaksiSept_2026_Update.csv",
    header=True, inferSchema=True
)
print("Jumlah baris dataset dari hdfs = ",df_dari_hdfs.count())
df.printSchema()
df_dari_hdfs.show(10)


Jumlah baris dataset dari hdfs =  1000
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...

In [35]:
##Menangani data yang kosong
#Mengecek jumlah data null
jumlahdatanull = df.filter((df["rating"].isNull())) #gunakan perintah .isNull() untuk memilih baris dari suatu kolom yang null
print("Jumlah rating kosong pada dataset = ",jumlahdatanull.count())

#Mengisi data null
df.na.fill(value = 0.0, subset="rating").show(10)



Jumlah rating kosong pada dataset =  204
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
|ORD-3005|2026-09-09 00:00:00|             Fash

In [41]:
##Menambahkan kolom
from pyspark.sql import functions as F #import functions ketika ingin menambahkan kondisi untuk tier
#Menambahkan kolom total pendapatan
df =df.withColumn("total_pendapatan", df["unit_terjual"]*df["harga_satuan"])
#Menambahkan kolom tier transaksi
df = df.withColumn("tier_transaksi", 
                   F.when(F.col("total_pendapatan")>500000, "Besar")
                   .otherwise("Kecil"))
df.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [79]:
##Analisa dengan groupBy
from pyspark.sql.functions import sum as spark_sum, count, avg, col,round
#Kategori paling tinggi
kategori = df.groupBy("Kategori").agg(
    spark_sum("total_pendapatan").alias("total_harga")
).orderBy(col("total_harga").desc())
kategori.show()

#Kota jumlah tier besar terbanyak
jum_tier = df.filter((df["tier_transaksi"]=="Besar")).groupBy("Kota").agg(
        count(col("tier_transaksi")).alias("tier")
).orderBy(col("tier").desc())
jum_tier.show()

#Rating masing-masing metode pembayaran
rating = df.groupBy("metode_pembayaran").agg(
    round(avg("rating"),2).alias("rating_pembayaran")
).orderBy(col("rating_pembayaran").desc())
rating.show()


+--------------------+-----------+
|            Kategori|total_harga|
+--------------------+-----------+
|        Rumah Tangga|  138665000|
|   Makanan & Minuman|  131890000|
|Kesehatan & Kecan...|  128595000|
|            Olahraga|  126650000|
|             Fashion|  124075000|
|          Elektronik|  110295000|
+--------------------+-----------+

+----------+----+
|      Kota|tier|
+----------+----+
|      Solo|  92|
|  Magelang|  78|
|   Kebumen|  78|
|Yogyakarta|  75|
| Purworejo|  66|
|  Semarang|  65|
+----------+----+

+-----------------+-----------------+
|metode_pembayaran|rating_pembayaran|
+-----------------+-----------------+
|              COD|             4.17|
|    Transfer Bank|             4.16|
|         E-Wallet|             4.14|
|     Kartu Kredit|             4.11|
+-----------------+-----------------+



In [107]:
#Menyimpan file ke hdfs dalam format csv baru.
#!hdfs dfs -mkdir /user/mahasiswa/tugas4/processed #buat direktori baru biar rapi
!hdfs dfs -ls /user/mahasiswa/tugas4
#!mv transaksi_september_2026.csv TransaksiSept_2026_Update.csv
!ls

Found 2 items
drwxr-xr-x   - celloz supergroup          0 2026-09-11 21:57 /user/mahasiswa/tugas4/processed
-rw-r--r--   1 celloz supergroup      83497 2026-09-11 18:20 /user/mahasiswa/tugas4/transaksi_september_2026.csv
drwxr-xr-x   - celloz 11 Sep 18:17  .ipynb_checkpoints
.rw-r--r-- 48k celloz 11 Sep 17:34  data_transaksi_ecommerce.csv
.rw-r--r-- 34k celloz 11 Sep 21:19  Modul_Praktikum_BigData_Pertemuan3.ipynb
.rw-r--r-- 37k celloz 11 Sep 19:16  Modul_Praktikum_BigData_Pertemuan4.ipynb
.rw-r--r-- 83k celloz 11 Sep 18:19  TransaksiSept_2026_Update.csv
.rw-r--r-- 17k celloz 11 Sep 21:59  Tugas4_2505060021_MarcelloKP.ipynb


In [110]:
df.write.mode("overwrite") \ #atau df.coalesce(1) write.mode 
    .option("header","true") \
    .csv("hdfs://localhost:9000/user/mahasiswa/tugas4/processed/TransaksiSept_2026_Update") 

!hdfs dfs -ls /user/mahasiswa/tugas4/processed/

Found 1 items
drwxr-xr-x   - celloz supergroup          0 2026-09-11 21:57 /user/mahasiswa/tugas4/processed/TransaksiSept_2026_Update


Gunakan perintah :

**df.write.mode("overwrite") \
    .option("header","true) \
    .csv("hdfs://localhost9000/path/yang/dituju/nama_file_baru")**

untuk menyimpan hasil olahan dari spark ke HDFS

Atau bisa pakai

**df.coalesce(1)** 

seperti diatas biar nantinya file csv nya tidak terbagi ke beberapa partisi, hal itu disebabkan karena spark memungkinkan proses penulisan data berjalan paralel dan cepat di banyak komputer sekaligus (kalo secara real) sehingga filenya dibagi menjadi kecil-kecil.

In [3]:
#menutup SparkSession
spark.stop()
print("SparkSession ditutup")

SparkSession ditutup
